In [ ]:
import os
import cv2
import hashlib
from pathlib import Path

In [ ]:
DATASET_PATH = r"C:\Users\HP\Desktop\data_set\Data\Garbage_Classification"
BLUR_THRESHOLD = 100
CLASSES = ["Battery", "Cardboard", "Clothes", "Glass", "Metal", "Paper", "Plastic"]

In [ ]:
def is_blurred(image_path, threshold=BLUR_THRESHOLD):
    img = cv2.imread(str(image_path))
    if img is None:
        return True
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    return variance < threshold

In [ ]:
def get_image_hash(image_path):
    with open(image_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

In [ ]:
def clean_dataset(dataset_path):
    seen_hashes = set()
    stats = {"blurred": 0, "duplicates": 0, "corrupted": 0, "kept": 0}

    for class_name in CLASSES:
        class_path = Path(dataset_path) / class_name
        if not class_path.exists():
            print(f" : {class_name}")
            continue

        print(f"\n بتنضف class: {class_name}")
        images = list(class_path.glob("*.jpg")) + \
                 list(class_path.glob("*.jpeg")) + \
                 list(class_path.glob("*.png"))

        for img_path in images:
            if is_blurred(img_path):
                print(f"   Blurred/Corrupted: {img_path.name}")
                img_path.unlink()
                stats["blurred"] += 1
                continue

            img_hash = get_image_hash(img_path)
            if img_hash in seen_hashes:
                print(f"   Duplicate: {img_path.name}")
                img_path.unlink()
                stats["duplicates"] += 1
                continue

            seen_hashes.add(img_hash)
            stats["kept"] += 1

    print("\n" + "="*40)
    print("📊 تقرير التنضيف:")
    print(f"   صور اتحتفظ بيها : {stats['kept']}")
    print(f"   Blurred/Corrupted: {stats['blurred']}")
    print(f"   Duplicates       : {stats['duplicates']}")
    print("="*40)

clean_dataset(DATASET_PATH)

In [1]:
import os
import shutil
import random
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter


In [2]:
SOURCE_PATH = r"C:\Users\HP\Desktop\data_set\Data\Garbage_Classification"
OUTPUT_PATH = r"C:\Users\HP\Desktop\NN_Data_set"

CLASSES = ["Battery", "Cardboard", "Clothes", 
           "Glass", "Metal", "Paper", "Plastic"]

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
RANDOM_SEED = 42


In [3]:
print(" عدد الصور في كل Class:")
print("-" * 35)
for cls in CLASSES:
    imgs = list(Path(SOURCE_PATH, cls).glob("*.*"))
    print(f"  {cls:<12}: {len(imgs)} صورة")
print("-" * 35)

 عدد الصور في كل Class:
-----------------------------------
  Battery     : 775 صورة
  Cardboard   : 254 صورة
  Clothes     : 389 صورة
  Glass       : 626 صورة
  Metal       : 687 صورة
  Paper       : 457 صورة
  Plastic     : 235 صورة
-----------------------------------


In [4]:
for split in ["train", "val", "test"]:
    for cls in CLASSES:
        Path(OUTPUT_PATH, split, cls).mkdir(parents=True, exist_ok=True)


In [5]:
def augment_image(img):
    ops = [
        lambda x: x.transpose(Image.FLIP_LEFT_RIGHT),
        lambda x: x.rotate(random.choice([90, 180, 270])),
        lambda x: ImageEnhance.Brightness(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: ImageEnhance.Contrast(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: x.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.0))),
    ]
    chosen = random.sample(ops, k=random.randint(1, 2))
    for op in chosen:
        img = op(img)
    return img


In [6]:
random.seed(RANDOM_SEED)
train_data = {}

train_counts = {
    cls: int(len(list(Path(SOURCE_PATH, cls).glob("*.*"))) * TRAIN_RATIO)
    for cls in CLASSES
}
target_train = max(train_counts.values())

for cls in CLASSES:
    imgs = list(Path(SOURCE_PATH, cls).glob("*.*"))
    random.shuffle(imgs)
    total = len(imgs)

    train_end = int(total * TRAIN_RATIO)
    val_end   = train_end + int(total * VAL_RATIO)

    train_imgs = imgs[:train_end]
    val_imgs   = imgs[train_end:val_end]
    test_imgs  = imgs[val_end:]

    for img in val_imgs:
        shutil.copy2(img, Path(OUTPUT_PATH, "val", cls, img.name))
    for img in test_imgs:
        shutil.copy2(img, Path(OUTPUT_PATH, "test", cls, img.name))
    for img in train_imgs:
        shutil.copy2(img, Path(OUTPUT_PATH, "train", cls, img.name))

    train_data[cls] = train_imgs
    print(f" {cls:<12}: train={len(train_imgs)} | "
          f"val={len(val_imgs)} | test={len(test_imgs)}")

print("split_done")

 Battery     : train=542 | val=116 | test=117
 Cardboard   : train=177 | val=38 | test=39
 Clothes     : train=272 | val=58 | test=59
 Glass       : train=438 | val=93 | test=95
 Metal       : train=480 | val=103 | test=104
 Paper       : train=319 | val=68 | test=70
 Plastic     : train=164 | val=35 | test=36
split_done


In [7]:
for cls, train_imgs in train_data.items():
    needed    = target_train - len(train_imgs)
    aug_count = 0
    aug_index = 0

    while aug_count < needed:
        src = train_imgs[aug_index % len(train_imgs)]
        try:
            img     = Image.open(src).convert("RGB")
            aug_img = augment_image(img)
            aug_name = f"aug_{aug_count}_{src.name}"
            aug_img.save(Path(OUTPUT_PATH, "train", cls, aug_name))
            aug_count += 1
        except Exception:
            pass
        aug_index += 1

    print(f" {cls:<12}: aug={aug_count} | "
          f"train_total={len(train_imgs) + aug_count}")

print("/n Augmentation_done ")

 Battery     : aug=0 | train_total=542
 Cardboard   : aug=365 | train_total=542
 Clothes     : aug=270 | train_total=542
 Glass       : aug=104 | train_total=542
 Metal       : aug=62 | train_total=542
 Paper       : aug=223 | train_total=542
 Plastic     : aug=378 | train_total=542
/n Augmentation_done 


In [8]:
orig_trains = sum(
    int(len(list(Path(SOURCE_PATH, cls).glob("*.*"))) * TRAIN_RATIO)
    for cls in CLASSES
)
orig_val  = sum(len(list(Path(OUTPUT_PATH, "val",  cls).glob("*.*"))) for cls in CLASSES)
orig_test = sum(len(list(Path(OUTPUT_PATH, "test", cls).glob("*.*"))) for cls in CLASSES)
aug_train = sum(len(list(Path(OUTPUT_PATH, "train", cls).glob("*.*"))) for cls in CLASSES)

total_orig = orig_trains + orig_val + orig_test

print("="*50)
print("📊 التقرير النهائي:")
print("-"*50)
print(f"  train : {aug_train} صورة  ({orig_trains/total_orig*100:.1f}% من الأصليين )")
print(f"  val   : {orig_val}  صورة  ({orig_val/total_orig*100:.1f}% من الأصليين )")
print(f"  test  : {orig_test} صورة  ({orig_test/total_orig*100:.1f}% من الأصليين )")
print("-"*50)
print(f"  Total : {aug_train + orig_val + orig_test} صورة (شاملة الـ aug)")
print("="*50)


📊 التقرير النهائي:
--------------------------------------------------
  train : 3794 صورة  (69.9% من الأصليين )
  val   : 511  صورة  (14.9% من الأصليين )
  test  : 520 صورة  (15.2% من الأصليين )
--------------------------------------------------
  Total : 4825 صورة (شاملة الـ aug)
